
# 応用問題 (集大成): MLP に本物の MNIST を学習させる

## 目標

AI の「学習」の中身が実は **行列積の繰り返し** であることを, **多層パーセプトロン (MLP)** を一から実装して体験する。**forward (順伝播) → 損失 → backprop (逆伝播) → 勾配降下 (更新)** のループを自分で書き, **本物の MNIST 手書き数字**を分類できるネットワークを学習させる。

学習した重みは, 推論専用の問題 `00_mnist_infer` がそのまま読み込んで使う。**「学習」も「推論」も中身は同じ行列積**であることを確かめよう。

ネットワーク (入力 784 → 隠れ層 128 → 出力 10):

- forward: `h = ReLU(W1 x + b1)` (128次元),  `o = W2 h + b2` (10次元),  `p = softmax(o)`
- 損失: 多クラスのクロスエントロピー
- backprop: `do = p - onehot(y)`,  `gW2 += do·hᵀ`, `gb2 += do`, `dh = (W2ᵀ do)·[h>0]`, `gW1 += dh·xᵀ`, `gb1 += dh`
- 更新 (ミニバッチ勾配降下): バッチ内のサンプルにわたって勾配を**総和**し, `W -= lr·(勾配/バッチサイズ)`。

## 課題

データは NumPy 標準の **`.npy` 形式** (ヘッダ + 生バイナリ) で `data/` に用意してある。世の中で配布されている MNIST (`mnist.npz`) から取り出した訓練画像である:

- `data/x_train.npy` : 訓練画像 16000 枚 (各 784 画素, uint8 で 0..255)
- `data/y_train.npy` : 正解ラベル (int32, 0..9)

`.npy` の読み書き関数 `read_npy` / `write_npy` はソース内に用意済みなので, **入出力を書く必要はない** (並列化に集中せよ)。

各ミニバッチで一番重いのは **バッチ内の全サンプルにわたる forward + backprop の O(バッチ·HID·IN)**。各サンプルの寄与は独立なので並列化できる。ただし勾配配列 `gW1, gb1, gW2, gb2` への加算は競合するので, **配列 reduction** で安全に総和する:

- C++: `#pragma omp parallel for reduction(+:loss,correct,gb2[:OUT],gW2[:OUT*HID],gb1[:HID],gW1[:HID*IN])`
- Fortran: `!$omp parallel do private(...) reduction(+:loss,correct,gb2,gW2,gb1,gW1)` … `!$omp end parallel do` (サンプルごとの一時配列 `h,o,dout` は `private`)

これが `TODO` の並列化箇所である。スレッド数を変えても結果 (正解率) はほぼ同じになることを確認せよ。

## コンパイルと実行

```
# C++
nvc++ -fast -mp=multicore mlp_train.cpp -o mlp_train.exe

# Fortran
nvfortran -fast -mp=multicore mlp_train.f90 -o mlp_train.exe
```

引数: エポック数 `E` (既定 20), 学習率 `lr` (既定 0.1), ミニバッチサイズ `BS` (既定 100)。

```
OMP_NUM_THREADS=4 ./mlp_train.exe 20 0.1 100
```

## 期待される結果

```
epoch    0: loss=1.16??, train acc=73.??%
epoch    5: loss=0.2???, train acc=92.??%
...
epoch   19: loss=0.1???, train acc=96.??%
最終: N=16000, HID=128, epochs=20, loss=0.1???, train acc=96.??%
elapsed = ... sec
重みを data/W1.npy, b1.npy, W2.npy, b2.npy に保存しました
```

- 学習が進むと **損失が下がり, 正解率が上がる**。終了時に学習済みの重みが `data/W1.npy` などに保存される。
- この重みを `00_mnist_infer` に渡すと, **未知のテスト画像を 9 割以上認識する** (汎化)。
- `OMP_NUM_THREADS` を増やすと `elapsed` が短くなる (台数効果)。正解率は本質的に同じになる。
- (発展) 内側の行列積を SIMD 化, あるいは GPU にオフロードして更に高速化できる。



# 1. ツールの読み込み
- AIチュータ及びジョブ投入ツールの読み込み (カーネル起動後に一度実行すればよい)
  - `heytutor` : `%%hey` でAIチュータに質問できるようになる (使い方は末尾を参照)
  - `wisteria_submit` : `%%bash_submit` (先頭に `#PJM ...` を書く) でジョブ投入できるようになる


In [ ]:
import heytutor
import wisteria_submit

# 2. C++ ベースコード

In [ ]:
%%writefile_ mlp_train.cpp
#include <cstdio>
#include <cstdlib>
#include <cstring>
#include <cmath>
#include <omp.h>

/* 多層パーセプトロン (MLP) を自分で学習させ, 本物の MNIST 手書き数字を分類する。
   ネットワーク: 入力 784 (28x28画像) -> 隠れ層 HID=128 (ReLU) -> 出力 10クラス。
   AI の「学習」の中身が行列積の繰り返しであることを体験する。

   forward:  h = ReLU(W1 x + b1),  o = W2 h + b2,  p = softmax(o)
   損失:     多クラスのクロスエントロピー
   backprop: do = p - onehot(y),  gW2 += do h^T, gb2 += do,
             dh = (W2^T do)・[h>0],  gW1 += dh x^T,  gb1 += dh
   更新:     ミニバッチ内の勾配を総和し, W -= lr * grad/batch。
   並列化対象は「ミニバッチ内の全サンプルにわたる勾配の和」(配列 reduction)。

   入出力はNumPy標準の .npy 形式 (ヘッダ + 生バイナリ):
   - 読み: data/x_train.npy (uint8 [N,784]), data/y_train.npy (int32 [N])
   - 書き: data/W1.npy, b1.npy, W2.npy, b2.npy (float64) -> 00_mnist_infer が読む
   read_npy / write_npy は下に用意してある (I/O は本問題の主眼ではない)。 */

/* ---- .npy 読み込み: 任意の数値型を double 配列に読み込む (C順, 1〜2次元) ---- */
static double * read_npy(const char * path, long shape[2], int * ndim) {
  FILE * f = fopen(path, "rb");
  if (!f) { printf("%s が開けません\n", path); exit(1); }
  unsigned char magic[10];
  if (fread(magic, 1, 10, f) != 10 || memcmp(magic, "\x93NUMPY", 6) != 0) {
    printf("%s は .npy ではありません\n", path); exit(1);
  }
  int hlen = magic[8] | (magic[9] << 8);           /* ヘッダ(辞書文字列)の長さ */
  char * hdr = (char *)malloc(hlen + 1);
  fread(hdr, 1, hlen, f); hdr[hlen] = '\0';
  /* dtype (例: '<f8','|u1','<i4','<i8') と shape をヘッダ文字列から取り出す */
  char descr[8] = {0};
  { char * q = strstr(hdr, "descr"); q = strchr(q, ':'); q = strchr(q, '\'') + 1;
    int i = 0; while (*q != '\'' && i < 7) descr[i++] = *q++; descr[i] = '\0'; }
  long s0 = 1, s1 = 1; *ndim = 1;
  char * sp = strstr(hdr, "shape"); sp = strchr(sp, '(') + 1;
  s0 = atol(sp);
  char * comma = strchr(sp, ',');
  /* "(N,)" は1次元, "(N, M)" は2次元 */
  char * after = comma + 1; while (*after == ' ') after++;
  if (*after != ')') { s1 = atol(after); *ndim = 2; } else { s1 = 1; *ndim = 1; }
  shape[0] = s0; shape[1] = s1;
  long n = s0 * (*ndim == 2 ? s1 : 1);
  free(hdr);

  double * out = (double *)malloc(sizeof(double) * n);
  if (!strcmp(descr, "<f8")) {
    fread(out, sizeof(double), n, f);
  } else if (!strcmp(descr, "<f4")) {
    float * t = (float *)malloc(sizeof(float) * n); fread(t, sizeof(float), n, f);
    for (long i = 0; i < n; i++) out[i] = t[i]; free(t);
  } else if (!strcmp(descr, "|u1")) {
    unsigned char * t = (unsigned char *)malloc(n); fread(t, 1, n, f);
    for (long i = 0; i < n; i++) out[i] = t[i]; free(t);
  } else if (!strcmp(descr, "<i4")) {
    int * t = (int *)malloc(sizeof(int) * n); fread(t, sizeof(int), n, f);
    for (long i = 0; i < n; i++) out[i] = t[i]; free(t);
  } else if (!strcmp(descr, "<i8")) {
    long long * t = (long long *)malloc(sizeof(long long) * n); fread(t, sizeof(long long), n, f);
    for (long i = 0; i < n; i++) out[i] = (double)t[i]; free(t);
  } else {
    printf("%s: 未対応の dtype %s\n", path, descr); exit(1);
  }
  fclose(f);
  return out;
}

/* ---- .npy 書き出し: double 配列を float64 の .npy として書く (C順, 1〜2次元) ---- */
static void write_npy(const char * path, const double * data, long s0, long s1) {
  FILE * f = fopen(path, "wb");
  if (!f) { printf("%s が書けません\n", path); exit(1); }
  char shape[64];
  if (s1 > 0) snprintf(shape, sizeof(shape), "(%ld, %ld)", s0, s1);
  else        snprintf(shape, sizeof(shape), "(%ld,)", s0);
  char dict[128];
  int dn = snprintf(dict, sizeof(dict),
                    "{'descr': '<f8', 'fortran_order': False, 'shape': %s, }", shape);
  int total = 10 + dn + 1;                     /* +1 は末尾の改行 */
  int pad = (64 - (total % 64)) % 64;          /* 全体を64の倍数に揃える */
  int hlen = dn + 1 + pad;
  unsigned char head[10] = {0x93,'N','U','M','P','Y',1,0,
                            (unsigned char)(hlen & 0xff),(unsigned char)(hlen >> 8)};
  fwrite(head, 1, 10, f);
  fwrite(dict, 1, dn, f);
  for (int i = 0; i < pad; i++) fputc(' ', f);
  fputc('\n', f);
  long n = s0 * (s1 > 0 ? s1 : 1);
  fwrite(data, sizeof(double), n, f);
  fclose(f);
}

/* 状態を持たない乱数 (初期値生成用): (seed,k) から [0,1)。 */
static inline double draw_rand01(long long seed, long long k) {
  const long long M = 2147483647LL;
  long long x = ((seed % M) * 2654435761LL + (k % M) + 1) % M;
  x = ((x ^ (x >> 16)) * 1812433253LL) % M;
  x = ((x ^ (x >> 13)) * 1664525LL)    % M;
  x =  (x ^ (x >> 16)) % M;
  return (double)x / (double)M;
}

int main(int argc, char ** argv) {
  int    E  = (argc > 1 ? atoi(argv[1]) : 20);    /* エポック数 */
  double lr = (argc > 2 ? atof(argv[2]) : 0.1);   /* 学習率 */
  int    BS = (argc > 3 ? atoi(argv[3]) : 100);   /* ミニバッチサイズ */
  const int IN = 784, HID = 128, OUT = 10;

  /* --- 訓練データの読み込み (画素 0..255 -> 0..1 に正規化) --- */
  long sx[2], sy[2]; int nd;
  double * Xu = read_npy("data/x_train.npy", sx, &nd);   /* [N,784] */
  double * yd = read_npy("data/y_train.npy", sy, &nd);   /* [N] */
  long N = sx[0];
  double * X = (double *)malloc(sizeof(double) * N * IN);
  int    * y = (int *)malloc(sizeof(int) * N);
  for (long i = 0; i < N * IN; i++) X[i] = Xu[i] / 255.0;
  for (long i = 0; i < N; i++)      y[i] = (int)yd[i];
  free(Xu); free(yd);

  /* --- パラメータ初期化 (He 初期化, バイアスは 0) --- */
  double * W1 = (double *)malloc(sizeof(double) * HID * IN);
  double * b1 = (double *)malloc(sizeof(double) * HID);
  double * W2 = (double *)malloc(sizeof(double) * OUT * HID);
  double * b2 = (double *)malloc(sizeof(double) * OUT);
  double s1 = sqrt(2.0 / IN), s2 = sqrt(2.0 / HID);
  for (long k = 0; k < (long)HID * IN; k++)  W1[k] = (draw_rand01(k, 1) - 0.5) * 2.0 * s1;
  for (int k = 0; k < HID; k++)              b1[k] = 0.0;
  for (long k = 0; k < (long)OUT * HID; k++) W2[k] = (draw_rand01(k, 2) - 0.5) * 2.0 * s2;
  for (int k = 0; k < OUT; k++)              b2[k] = 0.0;

  /* 勾配の総和を入れる配列 */
  double * gW1 = (double *)malloc(sizeof(double) * HID * IN);
  double * gb1 = (double *)malloc(sizeof(double) * HID);
  double * gW2 = (double *)malloc(sizeof(double) * OUT * HID);
  double * gb2 = (double *)malloc(sizeof(double) * OUT);

  double loss = 0.0; long correct = 0;
  double t0 = omp_get_wtime();
  for (int ep = 0; ep < E; ep++) {
    loss = 0.0; correct = 0;
    for (long b0 = 0; b0 < N; b0 += BS) {
      long b1n = (b0 + BS < N) ? b0 + BS : N;        /* バッチ [b0, b1n) */
      long m = b1n - b0;
      memset(gW1, 0, sizeof(double) * HID * IN);
      memset(gb1, 0, sizeof(double) * HID);
      memset(gW2, 0, sizeof(double) * OUT * HID);
      memset(gb2, 0, sizeof(double) * OUT);

      /* バッチ内の全サンプルにわたる forward + backprop。各サンプルの勾配寄与を総和。
         損失・正解数はスカラ reduction, 勾配は配列 reduction で競合を避ける。 */
      // TODO: バッチのループを配列 reduction で並列化せよ: #pragma omp parallel for reduction(+:loss,correct,gb2[:OUT],gW2[:OUT*HID],gb1[:HID],gW1[:HID*IN]).
      for (long i = b0; i < b1n; i++) {
        const double * x = &X[i * IN];
        double h[128];                          /* HID=128 */
        for (int k = 0; k < HID; k++) {         /* h = ReLU(W1 x + b1) */
          double z = b1[k];
          const double * w = &W1[(long)k * IN];
          for (int j = 0; j < IN; j++) z += w[j] * x[j];
          h[k] = (z > 0.0) ? z : 0.0;
        }
        double o[10], omax = -1e300;            /* o = W2 h + b2 */
        for (int c = 0; c < OUT; c++) {
          double z = b2[c];
          const double * w = &W2[(long)c * HID];
          for (int k = 0; k < HID; k++) z += w[k] * h[k];
          o[c] = z; if (z > omax) omax = z;
        }
        double sum = 0.0;                        /* p = softmax(o) */
        for (int c = 0; c < OUT; c++) { o[c] = exp(o[c] - omax); sum += o[c]; }
        int best = 0; double bestv = -1.0;
        for (int c = 0; c < OUT; c++) {
          o[c] /= sum;
          if (o[c] > bestv) { bestv = o[c]; best = c; }
        }
        loss -= log(o[y[i]] + 1e-12);
        if (best == y[i]) correct++;
        /* backprop: do = p - onehot(y) */
        double dout[10];
        for (int c = 0; c < OUT; c++) dout[c] = o[c] - (c == y[i] ? 1.0 : 0.0);
        for (int c = 0; c < OUT; c++) {
          gb2[c] += dout[c];
          double * gw = &gW2[(long)c * HID];
          for (int k = 0; k < HID; k++) gw[k] += dout[c] * h[k];
        }
        for (int k = 0; k < HID; k++) {          /* dh = (W2^T do)・[h>0] */
          if (h[k] <= 0.0) continue;
          double dh = 0.0;
          for (int c = 0; c < OUT; c++) dh += W2[(long)c * HID + k] * dout[c];
          gb1[k] += dh;
          double * gw = &gW1[(long)k * IN];
          const double * x = &X[i * IN];
          for (int j = 0; j < IN; j++) gw[j] += dh * x[j];
        }
      }

      /* 更新 (バッチ内勾配を平均して降下) */
      double sc = lr / (double)m;
      for (long k = 0; k < (long)HID * IN; k++)  W1[k] -= sc * gW1[k];
      for (int k = 0; k < HID; k++)              b1[k] -= sc * gb1[k];
      for (long k = 0; k < (long)OUT * HID; k++) W2[k] -= sc * gW2[k];
      for (int k = 0; k < OUT; k++)              b2[k] -= sc * gb2[k];
    }
    loss /= (double)N;
    if (ep % 5 == 0 || ep == E - 1)
      printf("epoch %4d: loss=%.4f, train acc=%.2f%%\n", ep, loss, 100.0 * correct / N);
  }
  double elapsed = omp_get_wtime() - t0;

  printf("最終: N=%ld, HID=%d, epochs=%d, loss=%.4f, train acc=%.2f%%\n",
         N, HID, E, loss, 100.0 * correct / N);
  printf("elapsed = %.3f sec\n", elapsed);

  /* --- 学習済みの重みを .npy で書き出す (00_mnist_infer が読む) --- */
  write_npy("data/W1.npy", W1, HID, IN);
  write_npy("data/b1.npy", b1, HID, 0);
  write_npy("data/W2.npy", W2, OUT, HID);
  write_npy("data/b2.npy", b2, OUT, 0);
  printf("重みを data/W1.npy, b1.npy, W2.npy, b2.npy に保存しました\n");

  free(X); free(y); free(W1); free(b1); free(W2); free(b2);
  free(gW1); free(gb1); free(gW2); free(gb2);
  return 0;
}

## 2-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=multicore mlp_train.cpp -o mlp_train_cpp.exe

## 2-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./mlp_train_cpp.exe

# 3. Fortran ベースコード

In [ ]:
%%writefile_ mlp_train.f90
! 多層パーセプトロン (MLP) を自分で学習させ, 本物の MNIST 手書き数字を分類する。
! ネットワーク: 入力 784 (28x28画像) -> 隠れ層 HID=128 (ReLU) -> 出力 10クラス。
! forward -> softmax クロスエントロピー損失 -> backprop -> ミニバッチ勾配降下 を繰り返す。
! 並列化対象は「ミニバッチ内の全サンプルにわたる勾配の和」(配列 reduction)。
! 入出力はNumPy標準の .npy 形式:
!   読み: data/x_train.npy (uint8 [N,784]), data/y_train.npy (int32 [N])
!   書き: data/W1.npy, b1.npy, W2.npy, b2.npy (float64) -> 00_mnist_infer が読む
! read_npy / write_npy は下のモジュールに用意してある (I/O は主眼ではない)。

module npy_io
contains
  ! .npy を読み, 中身を「保存順 (C順) のまま」flat な real(8) 配列 a(1:n) に入れる。
  subroutine read_npy(path, a, s0, s1, ndim)
    character(*), intent(in) :: path
    real(8), allocatable, intent(out) :: a(:)
    integer, intent(out) :: s0, s1, ndim
    integer :: u, hlen, p1, p2, ios
    integer(8) :: n, i
    character(len=10) :: magic
    character(len=:), allocatable :: hdr, sub
    character(len=8) :: descr
    integer(1), allocatable :: t1(:)
    integer(4), allocatable :: t4(:)
    integer(8), allocatable :: t8(:)
    real(4),    allocatable :: r4(:)

    open(newunit=u, file=path, access='stream', form='unformatted', &
         status='old', action='read')
    read(u) magic
    if (magic(1:6) /= char(147)//'NUMPY') stop '.npy ではありません'
    hlen = ichar(magic(9:9)) + 256*ichar(magic(10:10))
    allocate(character(len=hlen) :: hdr)
    read(u) hdr
    if      (index(hdr,"'<f8'") > 0) then; descr = '<f8'
    else if (index(hdr,"'<f4'") > 0) then; descr = '<f4'
    else if (index(hdr,"'|u1'") > 0) then; descr = '|u1'
    else if (index(hdr,"'<i4'") > 0) then; descr = '<i4'
    else if (index(hdr,"'<i8'") > 0) then; descr = '<i8'
    else; stop '未対応の dtype'; end if
    p1 = index(hdr, '('); p2 = index(hdr, ')')
    sub = hdr(p1+1:p2-1)
    do i = 1, len(sub); if (sub(i:i) == ',') sub(i:i) = ' '; end do
    read(sub, *, iostat=ios) s0, s1
    if (ios /= 0) then; ndim = 1; s1 = 1; read(sub, *) s0
    else;               ndim = 2; end if
    n = int(s0,8) * merge(int(s1,8), 1_8, ndim == 2)

    allocate(a(n))
    select case (trim(descr))
    case ('<f8'); read(u) a
    case ('<f4'); allocate(r4(n)); read(u) r4; a = real(r4,8)
    case ('|u1'); allocate(t1(n)); read(u) t1
                  do i = 1, n; a(i) = real(merge(int(t1(i))+256, int(t1(i)), t1(i) < 0), 8); end do
    case ('<i4'); allocate(t4(n)); read(u) t4; a = real(t4,8)
    case ('<i8'); allocate(t8(n)); read(u) t8; a = real(t8,8)
    end select
    close(u)
  end subroutine read_npy

  ! flat な real(8) 配列 a (C順) を float64 の .npy として書き出す。s1=0 なら1次元。
  subroutine write_npy(path, a, s0, s1)
    character(*), intent(in) :: path
    real(8), intent(in) :: a(:)
    integer, intent(in) :: s0, s1
    integer :: u, base, pad, hlen, i
    character(len=64) :: shp
    character(len=:), allocatable :: dict

    if (s1 > 0) then; write(shp,'(a,i0,a,i0,a)') '(', s0, ', ', s1, ')'
    else;             write(shp,'(a,i0,a)')      '(', s0, ',)'; end if
    dict = "{'descr': '<f8', 'fortran_order': False, 'shape': " // trim(shp) // ", }"
    base = 10 + len(dict) + 1                  ! +1 は末尾の改行
    pad  = mod(64 - mod(base, 64), 64)         ! 全体を64の倍数に揃える
    hlen = len(dict) + 1 + pad

    open(newunit=u, file=path, access='stream', form='unformatted', &
         status='replace', action='write')
    write(u) char(147)//'NUMPY'//char(1)//char(0)
    write(u) char(mod(hlen,256)), char(hlen/256)
    write(u) dict
    do i = 1, pad; write(u) ' '; end do
    write(u) char(10)
    write(u) a
    close(u)
  end subroutine write_npy

  ! 状態を持たない乱数 (初期値生成用): (seed,k) から [0,1)。
  function draw_rand01(seed, k) result(u)
    integer(8), intent(in) :: seed, k
    real(8) :: u
    integer(8), parameter :: M = 2147483647_8
    integer(8) :: x
    x = modulo(modulo(seed, M) * 2654435761_8 + modulo(k, M) + 1_8, M)
    x = modulo(ieor(x, ishft(x, -16)) * 1812433253_8, M)
    x = modulo(ieor(x, ishft(x, -13)) * 1664525_8,    M)
    x = modulo(ieor(x, ishft(x, -16)), M)
    u = real(x, 8) / real(M, 8)
  end function draw_rand01
end module npy_io

program mlp_train
  use npy_io
  use omp_lib
  implicit none
  ! flat な 0始まり配列で C 版と同じ並びにする (W1(k*IN+j)=W1[k][j] 等)
  integer, parameter :: IN = 784, HID = 128, OUT = 10
  character(len=32) :: arg
  integer :: E, ep, c, kk, s0, s1, nd, best
  integer(8) :: N, i, b0, b1n, m
  real(8) :: lr, loss, z, omax, ssum, sc, bestv, t0, elapsed, dh, sq1, sq2
  integer(8) :: correct
  real(8), allocatable :: X(:), W1(:), b1(:), W2(:), b2(:)
  real(8), allocatable :: gW1(:), gb1(:), gW2(:), gb2(:), flat(:)
  real(8), allocatable :: h(:), o(:), dout(:)
  integer, allocatable :: y(:)

  E = 20; lr = 0.1d0
  if (command_argument_count() >= 1) then; call get_command_argument(1, arg); read(arg,*) E;  end if
  if (command_argument_count() >= 2) then; call get_command_argument(2, arg); read(arg,*) lr; end if
  ! ミニバッチサイズ (既定 100)
  block
    integer :: BS
    BS = 100
    if (command_argument_count() >= 3) then; call get_command_argument(3, arg); read(arg,*) BS; end if

    ! --- 訓練データの読み込み (画素 0..255 -> 0..1) ---
    call read_npy("data/x_train.npy", flat, s0, s1, nd); N = s0
    allocate(X(0:N*IN-1)); X = flat / 255.0d0
    call read_npy("data/y_train.npy", flat, s0, s1, nd)
    allocate(y(0:N-1)); y = nint(flat(1:N))

    ! --- パラメータ初期化 (He 初期化) ---
    allocate(W1(0:HID*IN-1), b1(0:HID-1), W2(0:OUT*HID-1), b2(0:OUT-1))
    allocate(gW1(0:HID*IN-1), gb1(0:HID-1), gW2(0:OUT*HID-1), gb2(0:OUT-1))
    sq1 = sqrt(2.0d0/IN); sq2 = sqrt(2.0d0/HID)
    do i = 0, int(HID,8)*IN-1; W1(i) = (draw_rand01(i,1_8)-0.5d0)*2.0d0*sq1; end do
    b1 = 0.0d0
    do i = 0, int(OUT,8)*HID-1; W2(i) = (draw_rand01(i,2_8)-0.5d0)*2.0d0*sq2; end do
    b2 = 0.0d0

    allocate(h(0:HID-1), o(0:OUT-1), dout(0:OUT-1))
    loss = 0.0d0; correct = 0
    t0 = omp_get_wtime()
    do ep = 0, E - 1
       loss = 0.0d0; correct = 0
       do b0 = 0, N - 1, BS
          b1n = min(b0 + BS, N)               ! バッチ [b0, b1n)
          m = b1n - b0
          gW1 = 0.0d0; gb1 = 0.0d0; gW2 = 0.0d0; gb2 = 0.0d0

          ! バッチ内の全サンプルにわたる forward + backprop。勾配を総和する。
          ! 損失・正解数はスカラ reduction, 勾配は配列 reduction で競合を避ける。
          ! TODO: バッチのループを配列 reduction で並列化せよ: !$omp parallel do private(...) reduction(+:loss,correct,gb2,gW2,gb1,gW1) (h,o,dout は private)。
          do i = b0, b1n - 1
             do kk = 0, HID - 1               ! h = ReLU(W1 x + b1)
                z = b1(kk)
                do c = 0, IN - 1
                   z = z + W1(kk*IN+c) * X(i*IN+c)
                end do
                h(kk) = max(0.0d0, z)
             end do
             omax = -1d300                     ! o = W2 h + b2
             do c = 0, OUT - 1
                z = b2(c)
                do kk = 0, HID - 1
                   z = z + W2(c*HID+kk) * h(kk)
                end do
                o(c) = z; if (z > omax) omax = z
             end do
             ssum = 0.0d0                       ! softmax
             do c = 0, OUT - 1; o(c) = exp(o(c)-omax); ssum = ssum + o(c); end do
             best = 0; bestv = -1.0d0
             do c = 0, OUT - 1
                o(c) = o(c)/ssum
                if (o(c) > bestv) then; bestv = o(c); best = c; end if
             end do
             loss = loss - log(o(y(i)) + 1.0d-12)
             if (best == y(i)) correct = correct + 1
             ! backprop: do = p - onehot(y)
             do c = 0, OUT - 1; dout(c) = o(c) - merge(1.0d0,0.0d0,c==y(i)); end do
             do c = 0, OUT - 1
                gb2(c) = gb2(c) + dout(c)
                do kk = 0, HID - 1
                   gW2(c*HID+kk) = gW2(c*HID+kk) + dout(c)*h(kk)
                end do
             end do
             do kk = 0, HID - 1                 ! dh = (W2^T do)・[h>0]
                if (h(kk) <= 0.0d0) cycle
                dh = 0.0d0
                do c = 0, OUT - 1; dh = dh + W2(c*HID+kk)*dout(c); end do
                gb1(kk) = gb1(kk) + dh
                do c = 0, IN - 1
                   gW1(kk*IN+c) = gW1(kk*IN+c) + dh * X(i*IN+c)
                end do
             end do
          end do
          ! TODO: 上で始めた parallel do 領域を閉じる (!$omp end parallel do)。

          ! 更新 (バッチ内勾配を平均して降下)
          sc = lr / real(m, 8)
          W1 = W1 - sc*gW1; b1 = b1 - sc*gb1; W2 = W2 - sc*gW2; b2 = b2 - sc*gb2
       end do
       loss = loss / real(N, 8)
       if (mod(ep,5) == 0 .or. ep == E-1) &
          print "(a,i4,a,f7.4,a,f6.2,a)", "epoch ", ep, ": loss=", loss, &
                ", train acc=", 100.0d0*correct/N, "%"
    end do
    elapsed = omp_get_wtime() - t0

    print "(a,i0,a,i0,a,i0,a,f7.4,a,f6.2,a)", "最終: N=", N, ", HID=", HID, &
         ", epochs=", E, ", loss=", loss, ", train acc=", 100.0d0*correct/N, "%"
    print "(a,f0.3,a)", "elapsed = ", elapsed, " sec"
  end block

  ! --- 学習済みの重みを .npy で書き出す (00_mnist_infer が読む) ---
  call write_npy("data/W1.npy", W1, HID, IN)
  call write_npy("data/b1.npy", b1, HID, 0)
  call write_npy("data/W2.npy", W2, OUT, HID)
  call write_npy("data/b2.npy", b2, OUT, 0)
  print "(a)", "重みを data/W1.npy, b1.npy, W2.npy, b2.npy に保存しました"
end program mlp_train

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=multicore mlp_train.f90 -o mlp_train_f90.exe

## 3-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./mlp_train_f90.exe


# 4. 発展目標 (できる範囲で挑戦)
- この問題の基本は **マルチコア並列化** (`#pragma omp parallel for` / `!$omp parallel do` など)。まずはここまでを目指す。
- 余力があれば次にも挑戦:
  - **GPUで並列化**: コンパイルを `-mp=gpu` にして, 重いループに `#pragma omp target teams distribute parallel for` (+ 必要に応じて `map`) を付ける。
  - **SIMD化** (11/12章): 内側ループに `#pragma omp simd`, またはベクトル型。 **ILP向上** (13章): ベクトル長 `nl` の調整。
- どこまで速くできるか挑戦し, その効果を下の「性能を比べる」で可視化しよう。

# 5. 性能を比べる (任意)
- 各プログラムは主計算部分の所要時間を `elapsed = ... sec` の形で表示する。
- 構成を変えて (スレッド数, SIMDの有無, GPU など) 実行し, 得られた **経過秒** を下の `DATA` に「ラベルと秒」で並べて実行すると, 棒グラフと「基準(先頭)比のスピードアップ」が出る。
- 試した構成だけ書けばよい (`#` で始まる行は無視)。


In [ ]:
import matplotlib.pyplot as plt

# 各構成の (ラベル, 経過秒) を並べる。先頭の行を基準(スピードアップ=1)にする。
# 自分が実際に試した構成の数値に書き換える。
DATA = [
    ("serial",    1.00),
    ("omp-8",     0.20),
    # ("simd",      0.50),
    # ("simd+omp",  0.07),
    # ("gpu",       0.05),
]

labels = [d[0] for d in DATA]
secs   = [float(d[1]) for d in DATA]
speed  = [secs[0] / s for s in secs]                 # 先頭(基準)比のスピードアップ
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].bar(labels, secs)
ax[0].set_ylabel("elapsed (s)")
ax[0].set_title("time (lower = faster)")
ax[1].bar(labels, speed)
ax[1].set_ylabel("speedup vs " + labels[0])
ax[1].set_title("speedup (higher = faster)")
for a in ax:
    a.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()


# 6. AIチュータへの質問の仕方 (参考)
- 先頭で `import heytutor` 済みなら, セルに `%%hey` と書いて質問できる。
- ChatGPTなどと同様に自由に質問してよい。ただし「答えをそのまま教えて」などは自制すること。
- セル内で使える変数 (自動で展開される):
  - `{file:FILENAME}` : _FILENAME_ の中身 (例: `{file:problem.md}`, `{file:mlp_train.cpp}`)
  - `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前, ...
- 以下は質問例 (必要に応じてコピーして使う。Fortranなら `.cpp` を `.f90` に書き換える)。

## 6-1. 一般的な質問

In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 6-2. この問題に関するヒント

In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}

## 6-3. 困ったときのヘルプ
- コンパイル時や実行時のエラー直後に実行するとエラーに関するヘルプが得られる。

In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:mlp_train.cpp}

コマンドと実行結果:
{bash[-1]}

## 6-4. フィードバック
- 答えが出た後も, 無駄なところやより良いやり方がないかを聞くことを推奨。

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:mlp_train.cpp}